In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install monai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# test_pretrained.py
import os
import torch
import monai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd,
    NormalizeIntensityd, Spacingd, Orientationd, EnsureTyped, Resized
)
from monai.data import Dataset, DataLoader
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.utils import set_determinism

# Set deterministic behavior for reproducibility
set_determinism(seed=42)

# Configure paths
data_dir = "/content/drive/MyDrive/file_data"  # Update with actual path
model_path = "/content/model.pt"  # Path to pre-trained model

# Create test dataset from the CSV
df = pd.read_csv('/content/cohort(in).csv')
test_files = []

for _, row in df.iterrows():
    test_files.append({
        "image": os.path.join(data_dir, row['T2W_NIFTI']),
        "label": os.path.join(data_dir, row['Gland_NIFTI'])
    })

# Define test transforms - ADD RESIZE TRANSFORM to ensure consistent dimensions
test_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(
        keys=["image", "label"],
        pixdim=[0.5, 0.5, 0.5],
        mode=("bilinear", "nearest")
    ),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    # Add resize to ensure consistent dimensions
    Resized(
        keys=["image", "label"],
        spatial_size=(512, 512, 64),  # Match the model's expected output size
        mode=("bilinear", "nearest")
    ),
    ScaleIntensityd(keys=["image"], minv=0, maxv=1),
    NormalizeIntensityd(keys=["image"]),
    EnsureTyped(keys=["image", "label"])
])

# Create test dataset and loader
test_ds = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(test_ds, batch_size=1, num_workers=2)

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained model
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,  # Model has 3 output channels
    channels=[16, 32, 64, 128, 256, 512],
    strides=[2, 2, 2, 2, 2],
    num_res_units=4,
    act="PRELU",
    norm="BATCH",
    dropout=0.15
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# Define metrics
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Create directory for visualizations
os.makedirs('segmentation_results', exist_ok=True)

# Evaluation
with torch.no_grad():
    dice_scores = []

    for idx, test_data in enumerate(test_loader):
        print(f"Processing case {idx+1}/{len(test_loader)}")
        test_inputs, test_labels = (
            test_data["image"].to(device),
            test_data["label"].to(device)
        )

        print(f"Input shape: {test_inputs.shape}")
        print(f"Label shape: {test_labels.shape}")

        # Make prediction
        test_outputs = model(test_inputs)
        print(f"Output shape: {test_outputs.shape}")

        # Apply softmax to get probabilities
        test_outputs = torch.softmax(test_outputs, dim=1)

        # Convert binary label to one-hot with 3 channels to match model output
        test_labels_binary = (test_labels > 0.5).float()
        test_labels_onehot = torch.zeros_like(test_outputs)
        test_labels_onehot[:, 0, ...] = 1 - test_labels_binary.squeeze(1)  # Background
        test_labels_onehot[:, 1, ...] = test_labels_binary.squeeze(1)      # Prostate
        # Channel 2 stays zeros (assumed to be another segment not in your binary data)

        # Get the most relevant channel from the model output (likely channel 1)
        relevant_output = test_outputs[:, 1:2, ...]  # Take channel 1 (prostate gland)
        relevant_label = test_labels_binary

        # Check dice directly on binary predictions
        pred_binary = (relevant_output > 0.5).float()

        # Calculate dice manually for verification
        intersection = torch.sum(pred_binary * relevant_label)
        union = torch.sum(pred_binary) + torch.sum(relevant_label)
        manual_dice = 2.0 * intersection / (union + 1e-5)

        # Also try MONAI's dice metric
        try:
            dice_metric(y_pred=pred_binary, y=relevant_label)
            current_dice = dice_metric.aggregate().item()
            dice_metric.reset()
        except Exception as e:
            print(f"Error computing metrics with MONAI: {e}")
            current_dice = manual_dice.item()

        dice_scores.append(current_dice)
        print(f"  - Dice: {current_dice:.4f}")

        # Visualization - Create mid-axial view
        slice_idx = test_inputs.shape[4] // 2  # Middle slice in z-dimension

        # Extract middle slices
        input_slice = test_inputs[0, 0, :, :, slice_idx].cpu().numpy()
        label_slice = test_labels[0, 0, :, :, slice_idx].cpu().numpy()
        pred_slice = pred_binary[0, 0, :, :, slice_idx].cpu().numpy()

        # Create a figure with three subplots
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Display input image
        axes[0].imshow(input_slice, cmap='gray')
        axes[0].set_title('Input T2 Image')
        axes[0].axis('off')

        # Display ground truth
        axes[1].imshow(input_slice, cmap='gray')
        axes[1].imshow(label_slice, cmap='Reds', alpha=0.3)
        axes[1].set_title('Ground Truth Segmentation')
        axes[1].axis('off')

        # Display prediction
        axes[2].imshow(input_slice, cmap='gray')
        axes[2].imshow(pred_slice, cmap='Blues', alpha=0.3)
        axes[2].set_title(f'Model Prediction (Dice: {current_dice:.4f})')
        axes[2].axis('off')

        # Save figure
        plt.tight_layout()
        plt.savefig(f'segmentation_results/case_{idx+1}_slice_{slice_idx}.png', dpi=150)
        plt.close()

        # Create mid-sagittal view
        sag_slice_idx = test_inputs.shape[2] // 2  # Middle slice in x-dimension

        # Extract middle slices (sagittal view)
        input_sag = test_inputs[0, 0, sag_slice_idx, :, :].cpu().numpy()
        label_sag = test_labels[0, 0, sag_slice_idx, :, :].cpu().numpy()
        pred_sag = pred_binary[0, 0, sag_slice_idx, :, :].cpu().numpy()

        # Create a figure with three subplots (sagittal view)
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Display input image
        axes[0].imshow(input_sag, cmap='gray')
        axes[0].set_title('Input T2 Image (Sagittal)')
        axes[0].axis('off')

        # Display ground truth
        axes[1].imshow(input_sag, cmap='gray')
        axes[1].imshow(label_sag, cmap='Reds', alpha=0.3)
        axes[1].set_title('Ground Truth Segmentation')
        axes[1].axis('off')

        # Display prediction
        axes[2].imshow(input_sag, cmap='gray')
        axes[2].imshow(pred_sag, cmap='Blues', alpha=0.3)
        axes[2].set_title(f'Model Prediction')
        axes[2].axis('off')

        # Save figure
        plt.tight_layout()
        plt.savefig(f'segmentation_results/case_{idx+1}_sagittal_slice_{sag_slice_idx}.png', dpi=150)
        plt.close()

        # Also create 3D visualizations for selected cases (e.g., every 10th case)
        if idx % 10 == 0:
            # Create multiple slice visualizations
            for z_slice in range(0, test_inputs.shape[4], 8):  # Sample every 8 slices
                if z_slice >= test_inputs.shape[4]:
                    continue

                input_slice = test_inputs[0, 0, :, :, z_slice].cpu().numpy()
                label_slice = test_labels[0, 0, :, :, z_slice].cpu().numpy()
                pred_slice = pred_binary[0, 0, :, :, z_slice].cpu().numpy()

                # Create a figure with three subplots
                fig, axes = plt.subplots(1, 3, figsize=(18, 6))

                # Display input image
                axes[0].imshow(input_slice, cmap='gray')
                axes[0].set_title(f'Input T2 Image (Slice {z_slice})')
                axes[0].axis('off')

                # Display ground truth
                axes[1].imshow(input_slice, cmap='gray')
                axes[1].imshow(label_slice, cmap='Reds', alpha=0.3)
                axes[1].set_title('Ground Truth Segmentation')
                axes[1].axis('off')

                # Display prediction
                axes[2].imshow(input_slice, cmap='gray')
                axes[2].imshow(pred_slice, cmap='Blues', alpha=0.3)
                axes[2].set_title(f'Model Prediction')
                axes[2].axis('off')

                # Save figure
                plt.tight_layout()
                plt.savefig(f'segmentation_results/case_{idx+1}_3d_slice_{z_slice}.png', dpi=150)
                plt.close()

    # Calculate overall statistics
    if dice_scores:
        mean_dice = np.mean(dice_scores)
        std_dice = np.std(dice_scores)

        print("\nOverall Performance on D2 Dataset:")
        print(f"Dice Score: {mean_dice:.4f} ± {std_dice:.4f}")

        # Save results to CSV
        results_df = pd.DataFrame({
            'case_id': list(range(1, len(dice_scores) + 1)),
            'dice_score': dice_scores,
        })

        results_df.to_csv('pretrained_model_results.csv', index=False)
        print("Results saved to pretrained_model_results.csv")

        # Create a histogram of Dice scores
        plt.figure(figsize=(10, 6))
        plt.hist(dice_scores, bins=20, alpha=0.7, color='blue')
        plt.axvline(mean_dice, color='red', linestyle='dashed', linewidth=2, label=f'Mean Dice: {mean_dice:.4f}')
        plt.xlabel('Dice Score')
        plt.ylabel('Number of Cases')
        plt.title('Distribution of Dice Scores on D2 Dataset')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig('segmentation_results/dice_score_distribution.png', dpi=150)
        plt.close()
    else:
        print("No valid dice scores were calculated.")

Using device: cuda
Processing case 1/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.0000
Processing case 2/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.1875
Processing case 3/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.5977
Processing case 4/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.5208
Processing case 5/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1, 512, 512, 64])
Output shape: torch.Size([1, 3, 512, 512, 64])
  - Dice: 0.6671
Processing case 6/80
Input shape: torch.Size([1, 1, 512, 512, 64])
Label shape: torch.Size([1, 1

In [1]:
# finetune.py
import os
import torch
import monai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd,
    NormalizeIntensityd, Spacingd, Orientationd, EnsureTyped, Resized,
    RandAffined, RandGaussianNoised, RandRotate90d, RandFlipd
)
from monai.data import Dataset, DataLoader
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.utils import set_determinism
import time
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Set deterministic behavior for reproducibility
set_determinism(seed=42)

# Configure paths
data_dir = "/content/drive/MyDrive/file_data"  # Update with actual path
pretrained_model_path = "/content/model.pt"  # Path to pre-trained model
fine_tuned_model_path = "model_finetuned.pt"  # Where to save fine-tuned model

# Create a directory for visualizations and results
os.makedirs('finetuning_results', exist_ok=True)

# Create train-validation-test split with specific sizes
df = pd.read_csv('/content/cohort(in).csv')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle the data

train_size = 57
val_size = 11
test_size = 12

train_files = []
val_files = []
test_files = []

for i, row in df.iterrows():
    data_item = {
        "image": os.path.join(data_dir, row['T2W_NIFTI']),
        "label": os.path.join(data_dir, row['Gland_NIFTI'])
    }
    if i < train_size:
        train_files.append(data_item)
    elif i < train_size + val_size:
        val_files.append(data_item)
    else:
        test_files.append(data_item)

print(f"Training set: {len(train_files)} cases")
print(f"Validation set: {len(val_files)} cases")
print(f"Test set: {len(test_files)} cases")

# Define transforms for training - with data augmentation
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(
        keys=["image", "label"],
        pixdim=[0.5, 0.5, 0.5],
        mode=("bilinear", "nearest")
    ),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Resized(
        keys=["image", "label"],
        spatial_size=(512, 512, 64),  # Match the model's expected output size
        mode=("bilinear", "nearest")
    ),
    ScaleIntensityd(keys=["image"], minv=0, maxv=1),
    NormalizeIntensityd(keys=["image"]),
    # Data augmentation transforms
    RandAffined(
        keys=["image", "label"],
        prob=0.5,
        rotate_range=(0.05, 0.05, 0.05),
        scale_range=(0.1, 0.1, 0.1),
        mode=("bilinear", "nearest"),
        spatial_size=(512, 512, 64)
    ),
    RandGaussianNoised(keys=["image"], prob=0.3, mean=0.0, std=0.01),
    RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.5),
    RandRotate90d(keys=["image", "label"], prob=0.2, spatial_axes=(0, 1)),
    EnsureTyped(keys=["image", "label"])
])

# Define transforms for validation and testing - no augmentation
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(
        keys=["image", "label"],
        pixdim=[0.5, 0.5, 0.5],
        mode=("bilinear", "nearest")
    ),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Resized(
        keys=["image", "label"],
        spatial_size=(512, 512, 64),  # Match the model's expected output size
        mode=("bilinear", "nearest")
    ),
    ScaleIntensityd(keys=["image"], minv=0, maxv=1),
    NormalizeIntensityd(keys=["image"]),
    EnsureTyped(keys=["image", "label"])
])

# Create training, validation, and test datasets
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)
test_ds = Dataset(data=test_files, transform=val_transforms)  # Same transforms as validation

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained model
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,  # Original model has 3 output channels
    channels=[16, 32, 64, 128, 256, 512],
    strides=[2, 2, 2, 2, 2],
    num_res_units=4,
    act="PRELU",
    norm="BATCH",
    dropout=0.15
).to(device)

model.load_state_dict(torch.load(pretrained_model_path, map_location=device))
print("Pre-trained model loaded successfully")

# Define loss function and optimizer
loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=5, verbose=True)

# Define metrics
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Setup training variables
num_epochs = 50
best_metric = -1
best_metric_epoch = -1
val_interval = 1
epoch_loss_values = []
metric_values = []
lr_values = []

total_start = time.time()

# Training loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    # Training
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device)
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        # Convert binary label to match output channels
        labels_binary = (labels > 0.5).float()

        # Loss calculation
        loss = loss_function(outputs, labels_binary)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    lr_values.append(optimizer.param_groups[0]['lr'])

    print(f"Epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            val_dice_scores = []

            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["label"].to(device)
                )

                val_outputs = model(val_inputs)
                val_outputs = torch.softmax(val_outputs, dim=1)

                # Focus on relevant channel (prostate)
                val_relevant_output = val_outputs[:, 1:2, ...]
                val_pred = (val_relevant_output > 0.5).float()
                val_labels_binary = (val_labels > 0.5).float()

                # Calculate dice manually
                intersection = torch.sum(val_pred * val_labels_binary)
                union = torch.sum(val_pred) + torch.sum(val_labels_binary)
                val_dice = 2.0 * intersection / (union + 1e-5)
                val_dice_scores.append(val_dice.item())

            metric = np.mean(val_dice_scores)
            metric_values.append(metric)

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), fine_tuned_model_path)
                print("Saved new best model")

            print(f"Current epoch: {epoch + 1}, current Dice: {metric:.4f}")
            print(f"Best Dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

            # Update learning rate based on validation performance
            scheduler.step(epoch_loss)

    # Save intermediate visualizations every 10 epochs
    if (epoch + 1) % 10 == 0 or (epoch + 1) == num_epochs:
        # Plot loss and learning rate curves
        plt.figure("train", (12, 6))
        plt.subplot(1, 2, 1)
        plt.title("Epoch Average Loss")
        x = [i + 1 for i in range(len(epoch_loss_values))]
        y = epoch_loss_values
        plt.xlabel("epoch")
        plt.plot(x, y)

        plt.subplot(1, 2, 2)
        plt.title("Learning Rate")
        plt.xlabel("epoch")
        plt.plot(x, lr_values)

        plt.savefig(f"finetuning_results/epoch_{epoch+1}_loss_lr.png")
        plt.close()

        # Plot dice metric
        if len(metric_values) > 0:
            plt.figure("val", (6, 6))
            plt.title("Validation Dice")
            x = [val_interval * (i + 1) for i in range(len(metric_values))]
            y = metric_values
            plt.xlabel("epoch")
            plt.plot(x, y)
            plt.savefig(f"finetuning_results/epoch_{epoch+1}_dice.png")
            plt.close()

        # Save example predictions
        if val_loader:
            model.eval()
            with torch.no_grad():
                val_data = next(iter(val_loader))
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)

                val_outputs = model(val_inputs)
                val_outputs = torch.softmax(val_outputs, dim=1)

                # Get predictions
                val_relevant_output = val_outputs[:, 1:2, ...]
                val_pred = (val_relevant_output > 0.5).float()

                # Visualization of middle slice
                slice_idx = val_inputs.shape[4] // 2

                # Get slices
                input_slice = val_inputs[0, 0, :, :, slice_idx].cpu().numpy()
                label_slice = val_labels[0, 0, :, :, slice_idx].cpu().numpy()
                pred_slice = val_pred[0, 0, :, :, slice_idx].cpu().numpy()

                # Create figure with three subplots
                fig, axes = plt.subplots(1, 3, figsize=(18, 6))

                # Display input image
                axes[0].imshow(input_slice, cmap='gray')
                axes[0].set_title('Input T2 Image')
                axes[0].axis('off')

                # Display ground truth
                axes[1].imshow(input_slice, cmap='gray')
                axes[1].imshow(label_slice, cmap='Reds', alpha=0.3)
                axes[1].set_title('Ground Truth Segmentation')
                axes[1].axis('off')

                # Display prediction
                axes[2].imshow(input_slice, cmap='gray')
                axes[2].imshow(pred_slice, cmap='Blues', alpha=0.3)
                axes[2].set_title(f'Model Prediction (Epoch {epoch+1})')
                axes[2].axis('off')

                # Save figure
                plt.tight_layout()
                plt.savefig(f"finetuning_results/epoch_{epoch+1}_prediction.png", dpi=150)
                plt.close()

total_time = time.time() - total_start
print(f"Training completed in {total_time:.4f} seconds")
print(f"Best Dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

# Final evaluation of the fine-tuned model on the test set
print("\nFinal evaluation of fine-tuned model on test set...")

# Load the best model
model.load_state_dict(torch.load(fine_tuned_model_path, map_location=device))
model.eval()

with torch.no_grad():
    test_dice_scores = []

    for idx, test_data in enumerate(test_loader):
        print(f"Evaluating test case {idx+1}/{len(test_loader)}")
        test_inputs, test_labels = (
            test_data["image"].to(device),
            test_data["label"].to(device)
        )

        # Make prediction
        test_outputs = model(test_inputs)
        test_outputs = torch.softmax(test_outputs, dim=1)

        # Get relevant channel for prostate
        test_relevant_output = test_outputs[:, 1:2, ...]
        test_pred = (test_relevant_output > 0.5).float()
        test_labels_binary = (test_labels > 0.5).float()

        # Calculate dice
        intersection = torch.sum(test_pred * test_labels_binary)
        union = torch.sum(test_pred) + torch.sum(test_labels_binary)
        dice = 2.0 * intersection / (union + 1e-5)
        test_dice_scores.append(dice.item())

        # Visualization
        slice_idx = test_inputs.shape[4] // 2

        input_slice = test_inputs[0, 0, :, :, slice_idx].cpu().numpy()
        label_slice = test_labels[0, 0, :, :, slice_idx].cpu().numpy()
        pred_slice = test_pred[0, 0, :, :, slice_idx].cpu().numpy()

        # Create figure
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(input_slice, cmap='gray')
        axes[0].set_title('Input T2 Image')
        axes[0].axis('off')

        axes[1].imshow(input_slice, cmap='gray')
        axes[1].imshow(label_slice, cmap='Reds', alpha=0.3)
        axes[1].set_title('Ground Truth Segmentation')
        axes[1].axis('off')

        axes[2].imshow(input_slice, cmap='gray')
        axes[2].imshow(pred_slice, cmap='Blues', alpha=0.3)
        axes[2].set_title(f'Fine-tuned Model Prediction (Dice: {dice.item():.4f})')
        axes[2].axis('off')

        plt.tight_layout()
        plt.savefig(f"finetuning_results/test_case_{idx+1}.png", dpi=150)
        plt.close()

    # Calculate overall statistics
    mean_dice = np.mean(test_dice_scores)
    std_dice = np.std(test_dice_scores)

    print(f"\nTest Set Performance:")
    print(f"Dice Score: {mean_dice:.4f} ± {std_dice:.4f}")

    # Save results to CSV
    results_df = pd.DataFrame({
        'case_id': list(range(1, len(test_dice_scores) + 1)),
        'dice_score': test_dice_scores,
    })

    results_df.to_csv('finetuning_results/finetuned_model_test_results.csv', index=False)
    print("Results saved to finetuning_results/finetuned_model_test_results.csv")

    # Create histogram of Dice scores
    plt.figure(figsize=(10, 6))
    plt.hist(test_dice_scores, bins=10, alpha=0.7, color='green')
    plt.axvline(mean_dice, color='red', linestyle='dashed', linewidth=2, label=f'Mean Dice: {mean_dice:.4f}')
    plt.xlabel('Dice Score')
    plt.ylabel('Number of Cases')
    plt.title('Distribution of Dice Scores with Fine-tuned Model (Test Set)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('finetuning_results/finetuned_dice_score_distribution_test.png', dpi=150)
    plt.close()

# Also evaluate pre-trained model on test set for comparison
print("\nEvaluating pre-trained model on test set for comparison...")

# Load the pre-trained model
model.load_state_dict(torch.load(pretrained_model_path, map_location=device))
model.eval()

with torch.no_grad():
    pretrained_dice_scores = []

    for idx, test_data in enumerate(test_loader):
        print(f"Evaluating with pre-trained model: test case {idx+1}/{len(test_loader)}")
        test_inputs, test_labels = (
            test_data["image"].to(device),
            test_data["label"].to(device)
        )

        # Make prediction
        test_outputs = model(test_inputs)
        test_outputs = torch.softmax(test_outputs, dim=1)

        # Get relevant channel for prostate
        test_relevant_output = test_outputs[:, 1:2, ...]
        test_pred = (test_relevant_output > 0.5).float()
        test_labels_binary = (test_labels > 0.5).float()

        # Calculate dice
        intersection = torch.sum(test_pred * test_labels_binary)
        union = torch.sum(test_pred) + torch.sum(test_labels_binary)
        dice = 2.0 * intersection / (union + 1e-5)
        pretrained_dice_scores.append(dice.item())

    # Calculate overall statistics
    pretrained_mean_dice = np.mean(pretrained_dice_scores)
    pretrained_std_dice = np.std(pretrained_dice_scores)

    print(f"\nPre-trained Model Test Set Performance:")
    print(f"Dice Score: {pretrained_mean_dice:.4f} ± {pretrained_std_dice:.4f}")

    # Compare with fine-tuned model
    improvement = mean_dice - pretrained_mean_dice
    percentage_improvement = (improvement / pretrained_mean_dice) * 100

    print(f"\nImprovement with Fine-tuning:")
    print(f"Absolute Dice Improvement: {improvement:.4f}")
    print(f"Percentage Improvement: {percentage_improvement:.2f}%")

    # Create comparison bar chart
    plt.figure(figsize=(8, 6))
    models = ['Pre-trained', 'Fine-tuned']
    scores = [pretrained_mean_dice, mean_dice]
    errors = [pretrained_std_dice, std_dice]

    plt.bar(models, scores, yerr=errors, alpha=0.7, capsize=10)
    plt.ylabel('Mean Dice Score')
    plt.title('Comparison of Pre-trained vs. Fine-tuned Model Performance')
    plt.grid(True, alpha=0.3)
    plt.savefig('finetuning_results/model_comparison.png', dpi=150)
    plt.close()

    # Save comparison results
    comparison_df = pd.DataFrame({
        'model': ['Pre-trained', 'Fine-tuned'],
        'mean_dice': [pretrained_mean_dice, mean_dice],
        'std_dice': [pretrained_std_dice, std_dice],
        'improvement': [0, improvement],
        'percentage_improvement': [0, percentage_improvement]
    })

    comparison_df.to_csv('finetuning_results/model_comparison.csv', index=False)
    print("Comparison results saved to finetuning_results/model_comparison.csv")

Training set: 57 cases
Validation set: 11 cases
Test set: 12 cases
Using device: cuda
Pre-trained model loaded successfully

Epoch 1/50
1/29, train_loss: 1.8789
2/29, train_loss: 1.0904
3/29, train_loss: 0.9101
4/29, train_loss: 0.7885
5/29, train_loss: 0.7662
6/29, train_loss: 0.7507
7/29, train_loss: 0.7926
8/29, train_loss: 0.7962
9/29, train_loss: 0.7837
10/29, train_loss: 0.7617
11/29, train_loss: 0.9194
12/29, train_loss: 0.7252
13/29, train_loss: 0.7606
14/29, train_loss: 0.7780
15/29, train_loss: 0.7904
16/29, train_loss: 0.7529
17/29, train_loss: 0.7694
18/29, train_loss: 0.7613
19/29, train_loss: 0.6922
20/29, train_loss: 0.7764
21/29, train_loss: 0.7757
22/29, train_loss: 0.7046
23/29, train_loss: 0.7847
24/29, train_loss: 0.7756
25/29, train_loss: 0.7104
26/29, train_loss: 0.6757
27/29, train_loss: 0.7495
28/29, train_loss: 0.6700
29/29, train_loss: 0.6348
Epoch 1 average loss: 0.8112
Saved new best model
Current epoch: 1, current Dice: 0.1209
Best Dice: 0.1209 at epoch: 1
